# Video Inference

In [ ]:
import cv2
import torch
from pathlib import Path
from ultralytics import YOLO

#  Configuration 
VIDEO_PATH  = r"C:\Users\coigah\Documents\PAU\Naira\20250220_114607.mp4"
MODEL_PATH  = r"C:\Users\coigah\Documents\PAU\Naira\best_v10.pt"
CONF_THRESH = 0.50       # Minimum confidence threshold
IOU_THRESH  = 0.45       # NMS IoU threshold
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"



def run_video_inference():
    video_path = Path(VIDEO_PATH)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    # Output file sits next to the input video
    output_path = video_path.parent / f"{video_path.stem}_results{video_path.suffix}"

    print(f"Loading model : {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
    model.to(DEVICE)

    print(f"Opening video : {video_path}")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    print(f"Device        : {DEVICE}")
    print(f"Output        : {output_path}")
    print(f"Total frames  : {total}\n")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
        annotated = results.plot()          # BGR numpy array with boxes drawn
        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 50 == 0 or frame_idx == total:
            pct = (frame_idx / total * 100) if total else 0
            print(f"  Processed frame {frame_idx}/{total}  ({pct:.1f}%)")

    cap.release()
    writer.release()
    print(f"\nDone! Annotated video saved to:\n  {output_path}")


if __name__ == "__main__":
    run_video_inference()

# Image Inference

In [ ]:

import cv2
import torch
from pathlib import Path
from ultralytics import YOLO

#  Configuration 
INPUT_FOLDER = r"C:\Users\coigah\Documents\PAU\Naira\images"
MODEL_PATH   = r"C:\Users\coigah\Documents\PAU\Naira\best_v10.pt"
CONF_THRESH  = 0.25
IOU_THRESH   = 0.45
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".gif"}
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".m4v", ".mpeg", ".mpg"}


#  Helpers 

def process_image(model: YOLO, src: Path, dst: Path) -> None:
    """Run inference on a single image and save the annotated copy."""
    import cv2 as _cv2
    frame = _cv2.imread(str(src))
    if frame is None:
        print(f"  [WARN] Could not read image: {src.name}")
        return
    results   = model(frame, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
    annotated = results.plot()
    _cv2.imwrite(str(dst), annotated)
    print(f"  [IMG] {src.name}  →  {dst.name}")


def process_video(model: YOLO, src: Path, dst: Path) -> None:
    """Run inference frame-by-frame on a video and save the annotated copy."""
    cap = cv2.VideoCapture(str(src))
    if not cap.isOpened():
        print(f"  [WARN] Could not open video: {src.name}")
        return

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Keep the same container format as the source
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out_path = dst.with_suffix(".mp4")       # normalise to .mp4
    writer   = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    print(f"  [VID] {src.name}  ({total} frames) …")
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        results   = model(frame, conf=CONF_THRESH, iou=IOU_THRESH, verbose=False)[0]
        annotated = results.plot()
        writer.write(annotated)
        frame_idx += 1
        if frame_idx % 100 == 0:
            pct = (frame_idx / total * 100) if total else 0
            print(f"       … frame {frame_idx}/{total} ({pct:.1f}%)")

    cap.release()
    writer.release()
    print(f"        →  {out_path.name}")


#  Main 

def run_folder_inference():
    input_dir = Path(INPUT_FOLDER)
    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {input_dir}")

    results_dir = input_dir / "results"
    results_dir.mkdir(exist_ok=True)
    print(f"Results folder: {results_dir}\n")

    print(f"Loading model : {MODEL_PATH}")
    model = YOLO(MODEL_PATH)
    model.to(DEVICE)
    print(f"Device        : {DEVICE}\n")

    # Collect all supported files (non-recursive; add rglob for sub-folders)
    all_files = sorted(
        f for f in input_dir.iterdir()
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
    )

    if not all_files:
        print("No supported image or video files found in the input folder.")
        return

    images = [f for f in all_files if f.suffix.lower() in IMAGE_EXTS]
    videos = [f for f in all_files if f.suffix.lower() in VIDEO_EXTS]
    print(f"Found {len(images)} image(s) and {len(videos)} video(s).\n")

    #  Process images 
    if images:
        print(" Images ")
        for src in images:
            dst = results_dir / src.name
            process_image(model, src, dst)

    #  Process videos 
    if videos:
        print("\n Videos ")
        for src in videos:
            dst = results_dir / src.name
            process_video(model, src, dst)

    print(f"\nAll done!  Results saved to:\n  {results_dir}")


if __name__ == "__main__":
    run_folder_inference()